In [1]:
from minio import Minio

In [2]:
import pandas as pd
from io import BytesIO

In [3]:
client = Minio(
    "10.0.0.114:2711",
    access_key="intern",
    secret_key="12345678",
    secure=False
)

In [4]:
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name)

lqminh
minhmn
test-doanh


In [5]:
objects = client.list_objects("test-doanh", prefix="bronze/")
for object in objects:
    print(object.object_name, object.is_dir)
    print("=============")

bronze/churn_customer_snapshot/ True
bronze/churn_customers/ True
bronze/churn_marketing_interactions/ True
bronze/churn_order_items/ True
bronze/churn_orders/ True
bronze/churn_payments/ True
bronze/churn_product_usage/ True
bronze/churn_products/ True
bronze/churn_subscriptions/ True
bronze/churn_support_tickets/ True


In [6]:
objects = client.list_objects("test-doanh", prefix="bronze/", recursive=False)
for obj in objects:
    print(obj.object_name, obj.size)
    print("-"*50)


bronze/churn_customer_snapshot/ None
--------------------------------------------------
bronze/churn_customers/ None
--------------------------------------------------
bronze/churn_marketing_interactions/ None
--------------------------------------------------
bronze/churn_order_items/ None
--------------------------------------------------
bronze/churn_orders/ None
--------------------------------------------------
bronze/churn_payments/ None
--------------------------------------------------
bronze/churn_product_usage/ None
--------------------------------------------------
bronze/churn_products/ None
--------------------------------------------------
bronze/churn_subscriptions/ None
--------------------------------------------------
bronze/churn_support_tickets/ None
--------------------------------------------------


In [7]:
objects = client.list_objects("test-doanh", prefix="silver/", recursive=False)
for obj in objects:
    print(obj.object_name)

silver/churn_customers/
silver/churn_marketing_interactions/
silver/churn_order_items/
silver/churn_orders/
silver/churn_payments/
silver/churn_product_usage/
silver/churn_products/
silver/churn_subscriptions/
silver/churn_support_tickets/


In [8]:
objects = client.list_objects("test-doanh", prefix="silver/churn_products/", recursive=False)
object_names = []
for obj in objects:
    object_names.append(obj.object_name)

In [9]:
try:
    response = client.get_object("test-doanh", object_name=object_names[0])
    df = pd.read_parquet(BytesIO(response.read()))
    print(df.dtypes)
finally:
    response.close()
    response.release_conn()

category           str
price           object
product_id       int64
product_name       str
dtype: object


In [10]:
dfs = []
for name in object_names:
    if name.endswith(".parquet") == False:
        continue
    response = client.get_object("test-doanh", object_name=name)
    try:
        df = pd.read_parquet(BytesIO(response.read()))
        dfs.append(df)
    finally:
        response.close()
        response.release_conn()

product_df = pd.concat(dfs)


In [11]:
print(product_df.head(20))

      category      price  product_id    product_name
0       Laptop  408000.00           1       Laptop #1
1   Dien thoai  107000.00           2   Dien thoai #2
2     Gia dung  636000.00           3     Gia dung #3
3   Thoi trang  769000.00           4   Thoi trang #4
4   Thoi trang   44000.00           5   Thoi trang #5
5     Phu kien   83000.00           6     Phu kien #6
6       Laptop  342000.00           7       Laptop #7
7      Do choi  220000.00           8      Do choi #8
8       Laptop  296000.00           9       Laptop #9
9    Thuc pham  129000.00          10   Thuc pham #10
10        Sach  724000.00          11        Sach #11
11  Dien thoai  654000.00          12  Dien thoai #12
12  Dien thoai  321000.00          13  Dien thoai #13
13      Laptop  927000.00          14      Laptop #14
14  Thoi trang  480000.00          15  Thoi trang #15
15  Thoi trang  128000.00          16  Thoi trang #16
16     Do choi  435000.00          17     Do choi #17
17   Thuc pham  116000.00   

In [12]:
print(product_df.dtypes)

category           str
price           object
product_id       int64
product_name       str
dtype: object


In [13]:
client.list_buckets()

[Bucket(name='lqminh', creation_date=datetime.datetime(2026, 7, 27, 9, 18, 22, 39000, tzinfo=datetime.timezone.utc)),
 Bucket(name='minhmn', creation_date=datetime.datetime(2026, 7, 27, 8, 4, 58, 120000, tzinfo=datetime.timezone.utc)),
 Bucket(name='test-doanh', creation_date=datetime.datetime(2026, 7, 27, 7, 14, 33, 544000, tzinfo=datetime.timezone.utc))]

In [14]:
objects = client.list_objects("lqminh", prefix="silver/devdb/")
for obj in objects:
    print(obj.object_name)

silver/devdb/churn_customers/
silver/devdb/churn_marketing_interactions/
silver/devdb/churn_orders/
silver/devdb/churn_payments/
silver/devdb/churn_product_usage/
silver/devdb/churn_subscriptions/
silver/devdb/churn_support_tickets/


In [15]:
object_names = []
objects = client.list_objects("lqminh", prefix="silver/devdb/churn_customers/", recursive=True)
for obj in objects:
    print(obj.object_name)
    if obj.object_name.endswith(".parquet"):
        object_names.append(obj.object_name)



silver/devdb/churn_customers/_SUCCESS
silver/devdb/churn_customers/part-00000-ea3d4d33-ecf6-45bb-b223-7cb72ed0ee6c-c000.snappy.parquet
silver/devdb/churn_customers/part-00090-ea3d4d33-ecf6-45bb-b223-7cb72ed0ee6c-c000.snappy.parquet


In [16]:
dfs = []
for name in object_names:
    response = client.get_object("lqminh", name)
    try:
        df = pd.read_parquet(BytesIO(response.read()))
        dfs.append(df)
    finally:
        response.close()
        response.release_conn()
print(len(dfs))

2


In [17]:

churn_customers = pd.concat(dfs, ignore_index=True)
#churn_customers["birth_date"]=pd.to_datetime(churn_customers["birth_date"])
#churn_customers["signup_date"]=pd.to_datetime(churn_customers["signup_date"])
#churn_customers["closed_date"]=pd.to_datetime(churn_customers["closed_date"])
#churn_customers["total_amount"]=pd.to_numeric(churn_customers["total_amount"])
print(churn_customers.dtypes)
print(churn_customers.head(20))
print(type(churn_customers.loc[20, "birth_date"]))

customer_id                int32
gender                       str
birth_date        datetime64[ns]
region                       str
city                         str
signup_date       datetime64[ns]
account_status               str
closed_date       datetime64[ns]
last_login_at     datetime64[ns]
dtype: object
    customer_id gender birth_date      region         city signup_date  \
0             1      M 1991-05-19    Mien Nam  Ho Chi Minh  2025-03-24   
1             2      M 1979-12-31    Mien Bac   Quang Ninh  2025-05-15   
2             3      M 1987-07-07    Mien Nam      Can Tho  2025-12-16   
3             4      F 1978-12-13    Mien Nam      Can Tho  2025-03-23   
4             5      F 1990-08-13    Mien Bac       Ha Noi  2026-04-14   
5             6      F 2006-06-24  Mien Trung    Nha Trang  2025-01-08   
6             7      F 1982-04-08    Mien Bac     Nam Dinh  2026-04-27   
7             8      F 1969-07-29    Mien Nam  Ho Chi Minh  2024-02-14   
8             9      F 